In [ ]:
import requests, pandas as pd

pair = "ADAUSD"        # confirm via AssetPairs
interval = 1           # minutes: 1,5,15,30,60,240,1440,10080,21600
r = requests.get("https://api.kraken.com/0/public/OHLC",
                 params={"pair": pair, "interval": interval})
data = r.json()["result"][pair]
cols = ["time","open","high","low","close","vwap","volume","count"]
df = pd.DataFrame(data, columns=cols)
df["time"] = pd.to_datetime(df["time"], unit="s")
print(df.tail())

In [ ]:
import bokeh.plotting as bk
bk.output_notebook()

import numpy as np

In [ ]:
fig = bk.figure(
    width=1000, height=300,
    x_axis_label="Time",
    y_axis_label="Price [€]",
    x_axis_type="datetime"
)
fig.line(df["time"], df["open"])
fig.line(df["time"], df["close"], line_alpha=0.3)
bk.show(fig)

In [ ]:
val_hist = df["open"].apply(float).values

In [ ]:
dt = 0.1
tau = np.array([1.0, 5.0, 10.0, 20.0, 50.0, 100.0])
smoothed_val = np.array([val_hist[0].copy() for _ in tau])

smoothed_hist = [smoothed_val.copy()]

for val in val_hist[1:]:
    smoothed_val[:] += (val - smoothed_val) / tau * dt
    smoothed_hist.append(smoothed_val.copy())

smoothed_hist = np.stack(smoothed_hist)

In [ ]:
fig = bk.figure(
    width=1200, height=500, background_fill_color="#cccccc",
)
fig.line(np.arange(val_hist.size), val_hist, line_width=2)
for i, t in enumerate(tau):
    fig.line(np.arange(val_hist.size), smoothed_hist[:, i], line_color="red", line_alpha=0.7**i, legend_label=f"tau={t}")
bk.show(fig)

In [ ]:
fig = bk.figure(
    width=1200, height=500, background_fill_color="#cccccc",
)
for i, t in enumerate(tau):
    fig.line(np.arange(val_hist.size), (val_hist - smoothed_hist[:, i]) / smoothed_hist[:, i], line_color="red", line_alpha=0.7**i, legend_label=f"tau={t}")
bk.show(fig)

In [ ]:
val_hist.shape

In [ ]:
smoothed_hist.shape

In [ ]:
fig = bk.figure(
    width=1200, height=500, background_fill_color="#cccccc",
)
for i, t in enumerate(tau):
    fig.line(np.arange(val_hist.size), (val_hist - smoothed_hist[:, i]) / smoothed_hist[:, i], line_color="red", line_alpha=0.7**i, legend_label=f"tau={t}")
fig.line(np.arange(val_hist.size), ((val_hist - smoothed_hist.T) / smoothed_hist.T).mean(0), legend_label="sum")
bk.show(fig)